In [35]:
import numpy as np
import pandas as pd
import os

In [4]:

def add_cyclical_time_features(df: pd.DataFrame, timestamp_col: str) -> pd.DataFrame:

    ts = pd.to_datetime(df[timestamp_col])

    fractional_hour = ts.dt.hour + (ts.dt.minute / 60.0) + (ts.dt.second / 3600.0)
    radians_24h = 2 * np.pi * (fractional_hour / 24.0)

    df["Hour_X"] = np.cos(radians_24h)
    df["Hour_Y"] = np.sin(radians_24h)

    fractional_dow = ts.dt.dayofweek + (fractional_hour / 24.0)
    radians_dow = 2 * np.pi * (fractional_dow / 7.0)
    df["DoW_X"] = np.cos(radians_dow)
    df["DoW_Y"] = np.sin(radians_dow)

    df["is_weekend"] = (ts.dt.dayofweek >= 5).astype(int)

    return df

In [40]:
import pandas as pd

REFIT_MAPPINGS = {
    1: {
        "Appliance1": "Fridge",
        "Appliance2": "Chest Freezer",
        "Appliance3": "Upright Freezer",
        "Appliance4": "Tumble Dryer",
        "Appliance5": "Washing Machine",
        "Appliance6": "Dishwasher",
        "Appliance7": "Computer Site",
        "Appliance8": "Television Site",
        "Appliance9": "Electric Heater",
    },
    2: {
        "Appliance1": "Fridge-Freezer",
        "Appliance2": "Washing Machine",
        "Appliance3": "Dishwasher",
        "Appliance4": "Television",
        "Appliance5": "Microwave",
        "Appliance6": "Toaster",
        "Appliance7": "Hi-Fi",
        "Appliance8": "Kettle",
        "Appliance9": "Oven Extractor Fan",
    },
    3: {
        "Appliance1": "Toaster",
        "Appliance2": "Fridge-Freezer",
        "Appliance3": "Freezer",
        "Appliance4": "Tumble Dryer",
        "Appliance5": "Dishwasher",
        "Appliance6": "Washing Machine",
        "Appliance7": "Television",
        "Appliance8": "Microwave",
        "Appliance9": "Kettle",
    },
    4: {
        "Appliance1": "Fridge",
        "Appliance2": "Freezer",
        "Appliance3": "Fridge-Freezer",
        "Appliance4": "Washing Machine (1)",
        "Appliance5": "Washing Machine (2)",
        "Appliance6": "Computer Site",
        "Appliance7": "Television Site",
        "Appliance8": "Microwave",
        "Appliance9": "Kettle",
    },
    5: {
        "Appliance1": "Fridge-Freezer",
        "Appliance2": "Tumble Dryer",
        "Appliance3": "Washing Machine",
        "Appliance4": "Dishwasher",
        "Appliance5": "Computer Site",
        "Appliance6": "Television Site",
        "Appliance7": "Combination Microwave",
        "Appliance8": "Kettle",
        "Appliance9": "Toaster",
    },
    6: {
        "Appliance1": "Freezer (Utility Room)",
        "Appliance2": "Washing Machine",
        "Appliance3": "Dishwasher",
        "Appliance4": "MJY Computer",
        "Appliance5": "Television Site",
        "Appliance6": "Microwave",
        "Appliance7": "Kettle",
        "Appliance8": "Toaster",
        "Appliance9": "PGM Computer",
    },
    7: {
        "Appliance1": "Fridge",
        "Appliance2": "Freezer (Garage)",
        "Appliance3": "Freezer",
        "Appliance4": "Tumble Dryer",
        "Appliance5": "Washing Machine",
        "Appliance6": "Dishwasher",
        "Appliance7": "Television Site",
        "Appliance8": "Toaster",
        "Appliance9": "Kettle",
    },
    8: {
        "Appliance1": "Fridge",
        "Appliance2": "Freezer",
        "Appliance3": "Dryer",
        "Appliance4": "Washing Machine",
        "Appliance5": "Toaster",
        "Appliance6": "Computer",
        "Appliance7": "Television Site",
        "Appliance8": "Microwave",
        "Appliance9": "Kettle",
    },
    9: {
        "Appliance1": "Fridge-Freezer",
        "Appliance2": "Washer Dryer",
        "Appliance3": "Washing Machine",
        "Appliance4": "Dishwasher",
        "Appliance5": "Television Site",
        "Appliance6": "Microwave",
        "Appliance7": "Kettle",
        "Appliance8": "Hi-Fi",
        "Appliance9": "Electric Heater",
    },
    9: {
        "Appliance1": "Fridge-Freezer",
        "Appliance2": "Washer Dryer",
        "Appliance3": "Washing Machine",
        "Appliance4": "Dishwasher",
        "Appliance5": "Television Site",
        "Appliance6": "Microwave",
        "Appliance7": "Kettle",
        "Appliance8": "Hi-Fi",
        "Appliance9": "Electric Heater",
    },
    10: {
        "Appliance1": "Magimix Blender",
        "Appliance2": "Chest Freezer",
        "Appliance3": "Freezer",
        "Appliance4": "Washing Machine",
        "Appliance5": "Dishwasher",
        "Appliance6": "Television Site",
        "Appliance7": "Microwave",
        "Appliance8": "Toaster",
        "Appliance9": "Hi-Fi",
    },
    11: {
        "Appliance1": "Fridge",
        "Appliance2": "Fridge-Freezer",
        "Appliance3": "Washing Machine",
        "Appliance4": "Dishwasher",
        "Appliance5": "Computer Site",
        "Appliance6": "Microwave",
        "Appliance7": "Kettle",
        "Appliance8": "Router",
        "Appliance9": "Hi-Fi",
    },
    12: {
        "Appliance1": "Fridge-Freezer",
        "Appliance2": "Television Site",
        "Appliance3": "Microwave",
        "Appliance4": "Kettle",
        "Appliance5": "Toaster",
        "Appliance6": "Computer",
        "Appliance7": "Unknown",
        "Appliance8": "Unknown",
        "Appliance9": "Unknown",
    },
    13: {
        "Appliance1": "Television Site",
        "Appliance2": "Dishwasher",
        "Appliance3": "Washing Machine",
        "Appliance4": "Tumble Dryer",
        "Appliance5": "Microwave",
        "Appliance6": "Kettle",
        "Appliance7": "Toaster",
        "Appliance8": "Fridge",
        "Appliance9": "Freezer",
    },
    14: {
        "Appliance1": "Unknown",
        "Appliance2": "Unknown",
        "Appliance3": "Unknown",
        "Appliance4": "Unknown",
        "Appliance5": "Unknown",
        "Appliance6": "Unknown",
        "Appliance7": "Unknown",
        "Appliance8": "Unknown",
        "Appliance9": "Unknown",
    },
    15: {
        "Appliance1": "Fridge-Freezer",
        "Appliance2": "Tumble Dryer",
        "Appliance3": "Washing Machine",
        "Appliance4": "Dishwasher",
        "Appliance5": "Computer Site",
        "Appliance6": "Television Site",
        "Appliance7": "Microwave",
        "Appliance8": "Hi-Fi",
        "Appliance9": "Toaster",
    },
    16: {
        "Appliance1": "Fridge-Freezer (1)",
        "Appliance2": "Fridge-Freezer (2)",
        "Appliance3": "Washing Machine",
        "Appliance4": "Dishwasher",
        "Appliance5": "Computer Site",
        "Appliance6": "Television Site",
        "Appliance7": "Microwave",
        "Appliance8": "Kettle",
        "Appliance9": "Dehumidifier",
    },
    17: {
        "Appliance1": "Freezer",
        "Appliance2": "Fridge-Freezer",
        "Appliance3": "Washing Machine",
        "Appliance4": "Tumble Dryer",
        "Appliance5": "Computer Site",
        "Appliance6": "Television Site",
        "Appliance7": "Microwave",
        "Appliance8": "Kettle",
        "Appliance9": "Unknown",
    },
    18: {
        "Appliance1": "Fridge-Freezer",
        "Appliance2": "Freezer",
        "Appliance3": "Washing Machine",
        "Appliance4": "Dishwasher",
        "Appliance5": "Computer Site",
        "Appliance6": "Television Site",
        "Appliance7": "Microwave",
        "Appliance8": "Kettle",
        "Appliance9": "Toaster",
    },
    19: {
        "Appliance1": "Fridge-Freezer",
        "Appliance2": "Washing Machine",
        "Appliance3": "Television Site",
        "Appliance4": "Microwave",
        "Appliance5": "Kettle",
        "Appliance6": "Toaster",
        "Appliance7": "Breadmaker",
        "Appliance8": "Games Console",
        "Appliance9": "Hi-Fi",
    },
    20: {
        "Appliance1": "Fridge",
        "Appliance2": "Freezer",
        "Appliance3": "Washing Machine",
        "Appliance4": "Tumble Dryer",
        "Appliance5": "Dishwasher",
        "Appliance6": "Computer Site",
        "Appliance7": "Television Site",
        "Appliance8": "Microwave",
        "Appliance9": "Kettle",
    },
    21: {
        "Appliance1": "Fridge-Freezer",
        "Appliance2": "Washing Machine",
        "Appliance3": "Dishwasher",
        "Appliance4": "Television Site",
        "Appliance5": "Microwave",
        "Appliance6": "Kettle",
        "Appliance7": "Toaster",
        "Appliance8": "Vivarium",
        "Appliance9": "Pond Pump",
    },
}


def rename_refit_columns(
    df: pd.DataFrame, house_id: int, space_in_appliance_name: bool = True
) -> pd.DataFrame:

    cols_to_drop = ["Time", "Unix", "Issues"]
    df = df.drop(columns=cols_to_drop, errors="ignore")
   
    if house_id not in REFIT_MAPPINGS:
        raise ValueError(f"House {house_id} mapping is not defined.")

    mapping = REFIT_MAPPINGS[house_id]

    if space_in_appliance_name:
        mapping = {
            f"Appliance {k.replace('Appliance', '')}": v
            for k, v in mapping.items()
        }

    return df.rename(columns=mapping)



In [ ]:
def process_all_houses(data_dir: str = ".", output_dir: str = "./processed"):
    os.makedirs(output_dir, exist_ok=True)

    for house_id in range(1, 22):
        # Accommodates common naming styles ('House1.csv', 'House_1.csv', 'CLEAN_House1.csv')
        possible_filenames = [
            f"House{house_id}.csv",
            f"House_{house_id}.csv",
            f"CLEAN_House{house_id}.csv",
            f"CLEAN_House_{house_id}.csv",
        ]

        file_path = None
        for name in possible_filenames:
            candidate = os.path.join(data_dir, name)
            if os.path.exists(candidate):
                file_path = candidate
                break

        if not file_path:
            print(f"Skipping House {house_id}: File not found.")
            continue

        print(f"Processing House {house_id} from {file_path}...")

        # Load data
        df = pd.read_csv(file_path)

        # Standardize timestamp column naming
        ts_col = "Time" if "Time" in df.columns else "timestamp"
        if ts_col not in df.columns:
            # REFIT CSVs without headers typically have Time as the first column
            df.rename(columns={df.columns[0]: "Time"}, inplace=True)
            ts_col = "Time"

        # 1. Add continuous temporal features
        df = add_cyclical_time_features(df, timestamp_col=ts_col)

        # 2. Rename appliance columns and drop unwanted metadata
        df = rename_refit_columns(df, house_id=house_id)

        # Save processed output
        out_path = os.path.join(output_dir, f"House_{house_id}_processed.csv")
        df.to_csv(out_path, index=False)
        print(f"Saved: {out_path}")


process_all_houses(data_dir="/Users/simahamedifar/Documents/VSCode/Agentic_AI_HAR_Sima/CLEAN_REFIT_081116", output_dir="/Users/simahamedifar/Documents/VSCode/Agentic_AI_HAR_Sima/CLEAN_REFIT_081116/processed")
